# MLP-VaR Softplus: features dinamicas con ReLU base y SiLU - 2010-2024, w=1

Dos pruebas a partir del mejor MLP actual (`Softplus + ratios + shocks`) sin modificar la funcion de perdida ni el target:

1. `dynamic_features`: mantiene la arquitectura base y anade senales de regimen.
2. `dynamic_relu_layernorm`: usa las mismas features y cambia la arquitectura a ReLU + LayerNorm + Dropout.
3. `dynamic_silu_layernorm`: usa las mismas features y cambia la arquitectura a SiLU + LayerNorm + Dropout.

La perdida sigue siendo exactamente pinball sobre `target` en escala original con `w=1`.

In [1]:
import copy
import math
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from scipy.stats import chi2
from torch.utils.data import DataLoader, TensorDataset

try:
    from tqdm.auto import tqdm
except ModuleNotFoundError:
    def tqdm(iterable, *args, **kwargs):
        return iterable

torch.set_num_threads(1)
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 220)

c:\Users\trgglg\OneDrive - gmv.com\projects_w\TFG-GONZALO-LOPEZ-GARCIA-2026\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DATA_DIR = Path("../data")
if not DATA_DIR.exists():
    DATA_DIR = Path("data")
DATASET_PATH = DATA_DIR / "dataset_tfg.pkl"

ALPHA = 0.99
SIG = 0.05
W_LOSS = 1
WINDOW = 900
RETRAIN_EVERY = 1
SEED = 42
EVAL_START = "2010-01-01"
EVAL_END = "2024-12-31"

EXPERIMENTS = [
    {
        "key": "dynamic_features",
        "label": "Softplus + dynamic features",
        "architecture": "base_relu",
        "prefix": "softplus_dynamic_features_2010_2024_w1",
        "pred_file": "nn_softplus_dynamic_features_2010_2024_w1.csv",
    },
    {
        "key": "dynamic_relu_layernorm",
        "label": "Softplus + dynamic features + ReLU LayerNorm",
        "architecture": "relu_layernorm",
        "prefix": "softplus_dynamic_relu_layernorm_2010_2024_w1",
        "pred_file": "nn_softplus_dynamic_relu_layernorm_2010_2024_w1.csv",
    },
    {
        "key": "dynamic_silu_layernorm",
        "label": "Softplus + dynamic features + SiLU LayerNorm",
        "architecture": "silu_layernorm",
        "prefix": "softplus_dynamic_silu_layernorm_2010_2024_w1",
        "pred_file": "nn_softplus_dynamic_silu_layernorm_2010_2024_w1.csv",
    },
]

## Carga de datos y features dinamicas

In [3]:
dataset = pd.read_pickle(DATASET_PATH).copy().sort_index()
assert isinstance(dataset.index, pd.DatetimeIndex), "El indice debe ser DatetimeIndex"
assert "target" in dataset.columns, "Falta target"
assert "rp_lag_0" in dataset.columns, "Falta rp_lag_0"
assert "vol_20" in dataset.columns, "Falta vol_20"
assert "vol_60" in dataset.columns, "Falta vol_60"

# Senales del modelo 2: ratios de volatilidad y shocks relativos.
eps = 1e-8
abs_r = dataset["rp_lag_0"].abs()
vol_5 = dataset["rp_lag_0"].rolling(5).std(ddof=0)
vol_10 = dataset["rp_lag_0"].rolling(10).std(ddof=0)
vol_20_realized = dataset["rp_lag_0"].rolling(20).std(ddof=0)
vol_20_ref = dataset["vol_20"].abs()
vol_60_ref = dataset["vol_60"].abs()

# Base del mejor MLP anterior.
dataset["vol_5_ratio"] = vol_5 / (vol_20_realized + eps)
dataset["vol_10_ratio"] = vol_10 / (vol_20_realized + eps)
dataset["shock_1"] = abs_r / (vol_20_realized + eps)
dataset["shock_5"] = abs_r.rolling(5).max() / (vol_20_realized + eps)

# Nuevas features: regimen y persistencia reciente, sin cambiar target ni perdida.
dataset["vol_20_ratio_60"] = vol_20_ref / (vol_60_ref + eps)
dataset["vol_20_ratio_252"] = vol_20_ref / (vol_20_ref.rolling(252).median() + eps)
dataset["abs_ret_sum_5_ratio"] = abs_r.rolling(5).sum() / (vol_20_realized + eps)
dataset["abs_ret_sum_10_ratio"] = abs_r.rolling(10).sum() / (vol_20_realized + eps)
dataset["max_abs_ret_10_ratio"] = abs_r.rolling(10).max() / (vol_20_realized + eps)

dataset = dataset.replace([np.inf, -np.inf], np.nan).dropna().copy()

feature_cols = [c for c in dataset.columns if c != "target"]
new_features = [
    "vol_5_ratio", "vol_10_ratio", "shock_1", "shock_5",
    "vol_20_ratio_60", "vol_20_ratio_252",
    "abs_ret_sum_5_ratio", "abs_ret_sum_10_ratio", "max_abs_ret_10_ratio",
]

print("Dataset:", dataset.shape)
print("Rango:", dataset.index.min().date(), "->", dataset.index.max().date())
print("Features:", len(feature_cols))
print("Nuevas features incluidas:", [c for c in new_features if c in feature_cols])
print("target mean/std/min/max:", dataset["target"].mean(), dataset["target"].std(), dataset["target"].min(), dataset["target"].max())

assert len(feature_cols) == 31, f"Se esperaban 31 features, hay {len(feature_cols)}"
assert all(c in feature_cols for c in new_features), "Faltan features nuevas"
assert dataset["target"].abs().quantile(0.999) < 0.2, "El target parece tener escala inesperada"

Dataset: (4680, 32)
Rango: 2006-04-05 -> 2024-12-27
Features: 31
Nuevas features incluidas: ['vol_5_ratio', 'vol_10_ratio', 'shock_1', 'shock_5', 'vol_20_ratio_60', 'vol_20_ratio_252', 'abs_ret_sum_5_ratio', 'abs_ret_sum_10_ratio', 'max_abs_ret_10_ratio']
target mean/std/min/max: -0.00017689210141721946 0.00751750450881661 -0.07361457194412012 0.0784301570334365


## Metricas de backtesting

In [4]:
def hit_series(real_losses, var_preds):
    real_losses = np.asarray(real_losses, dtype=float).reshape(-1)
    var_preds = np.asarray(var_preds, dtype=float).reshape(-1)
    m = np.isfinite(real_losses) & np.isfinite(var_preds)
    return real_losses[m], var_preds[m], (real_losses[m] > var_preds[m]).astype(int)


def tick_loss(real_losses, var_preds, alpha=0.99):
    real_losses, var_preds, _ = hit_series(real_losses, var_preds)
    under = np.maximum(real_losses - var_preds, 0.0)
    over = np.maximum(var_preds - real_losses, 0.0)
    return float(np.mean(alpha * under + (1 - alpha) * over))


def winkler_score(real_losses, var_preds, alpha=0.99):
    real_losses, var_preds, _ = hit_series(real_losses, var_preds)
    exceedance = np.maximum(real_losses - var_preds, 0.0)
    return float(np.mean(var_preds + (2.0 / alpha) * exceedance))


def kupiec_test(real_losses, var_preds, alpha=0.99):
    _, _, I = hit_series(real_losses, var_preds)
    T, x = len(I), int(I.sum())
    p = 1 - alpha
    if T == 0 or x == 0 or x == T:
        return {"LRuc": np.nan, "p_uc": np.nan}
    p_hat = x / T
    LRuc = -2 * np.log(((1 - p) ** (T - x) * p ** x) / ((1 - p_hat) ** (T - x) * p_hat ** x))
    return {"LRuc": LRuc, "p_uc": 1 - chi2.cdf(LRuc, df=1)}


def christoffersen_cc_test(real_losses, var_preds, alpha=0.99):
    _, _, I = hit_series(real_losses, var_preds)
    T = len(I)
    if T < 2:
        return {"LRind": np.nan, "p_ind": np.nan, "LRcc": np.nan, "p_cc": np.nan}
    n00 = n01 = n10 = n11 = 0
    for t in range(1, T):
        pair = (I[t - 1], I[t])
        if pair == (0, 0):
            n00 += 1
        elif pair == (0, 1):
            n01 += 1
        elif pair == (1, 0):
            n10 += 1
        else:
            n11 += 1
    if (n00 + n01) == 0 or (n10 + n11) == 0:
        LRind, p_ind = np.nan, np.nan
    else:
        pi01 = n01 / (n00 + n01)
        pi11 = n11 / (n10 + n11)
        pi = (n01 + n11) / (n00 + n01 + n10 + n11)
        L0 = ((1 - pi) ** (n00 + n10)) * (pi ** (n01 + n11))
        L1 = ((1 - pi01) ** n00) * (pi01 ** n01) * ((1 - pi11) ** n10) * (pi11 ** n11)
        if L0 <= 0 or L1 <= 0:
            LRind, p_ind = np.nan, np.nan
        else:
            LRind = -2 * np.log(L0 / L1)
            p_ind = 1 - chi2.cdf(LRind, df=1)
    kup = kupiec_test(real_losses, var_preds, alpha=alpha)
    LRuc = kup["LRuc"]
    if np.isnan(LRuc) or np.isnan(LRind):
        return {"LRind": LRind, "p_ind": p_ind, "LRcc": np.nan, "p_cc": np.nan}
    LRcc = LRuc + LRind
    return {"LRind": LRind, "p_ind": p_ind, "LRcc": LRcc, "p_cc": 1 - chi2.cdf(LRcc, df=2)}


def dq_test(real_losses, var_preds, alpha=0.99, lags=4):
    _, _, I = hit_series(real_losses, var_preds)
    T = len(I)
    p = 1 - alpha
    if T <= lags + 5:
        return {"DQ": np.nan, "p_dq": np.nan}
    hit = I - p
    y = hit[lags:]
    X = np.column_stack([np.ones(len(y))] + [hit[lags - j:T - j] for j in range(1, lags + 1)])
    XtX = X.T @ X
    if np.linalg.matrix_rank(XtX) < XtX.shape[0]:
        return {"DQ": np.nan, "p_dq": np.nan}
    beta = np.linalg.inv(XtX) @ X.T @ y
    resid = y - X @ beta
    sigma2 = (resid @ resid) / len(y)
    if sigma2 <= 0:
        return {"DQ": np.nan, "p_dq": np.nan}
    DQ = float((beta.T @ XtX @ beta) / sigma2)
    return {"DQ": DQ, "p_dq": 1 - chi2.cdf(DQ, df=X.shape[1])}

In [5]:
def describe_scale(name, x):
    x = np.asarray(x, dtype=float)
    print(f"\n{name}")
    print("min:", np.nanmin(x))
    print("max:", np.nanmax(x))
    print("mean:", np.nanmean(x))
    print("p95:", np.nanpercentile(x, 95))
    print("p99:", np.nanpercentile(x, 99))
    print("p99.9:", np.nanpercentile(x, 99.9))


def plausibility_report(df, var_col="VaR_pred", loss_col="loss_real"):
    describe_scale("loss_real", df[loss_col].values)
    describe_scale(var_col, df[var_col].values)
    n_negative = int((df[var_col] < 0).sum())
    max_date = df[var_col].idxmax()
    min_date = df[var_col].idxmin()
    print("n_negative_var:", n_negative)
    print("max VaR:", max_date, float(df.loc[max_date, var_col]))
    print("min VaR:", min_date, float(df.loc[min_date, var_col]))


def evaluate_var_model(df, model_label, alpha=0.99, sig=0.05):
    real_losses = df["loss_real"].values
    var_preds = df["VaR_pred"].values
    _, _, I = hit_series(real_losses, var_preds)
    T = len(I)
    violations = int(I.sum())
    violation_rate = violations / T
    kup = kupiec_test(real_losses, var_preds, alpha=alpha)
    cc = christoffersen_cc_test(real_losses, var_preds, alpha=alpha)
    dq = dq_test(real_losses, var_preds, alpha=alpha, lags=4)
    df_2020 = df.loc["2020"] if (df.index.year == 2020).any() else pd.DataFrame(columns=df.columns)
    row = {
        "Model": model_label,
        "w": W_LOSS,
        "T": T,
        "Expected Viol.": (1 - alpha) * T,
        "Violations": violations,
        "Violation Rate": violation_rate,
        "VR Gap": abs(violation_rate - (1 - alpha)),
        "Coverage Bias": violation_rate - (1 - alpha),
        "Tick Loss": tick_loss(real_losses, var_preds, alpha=alpha),
        "Winkler Score": winkler_score(real_losses, var_preds, alpha=alpha),
        "Mean VaR Level": float(np.nanmean(var_preds)),
        "Max VaR Level": float(np.nanmax(var_preds)),
        "Min VaR Level": float(np.nanmin(var_preds)),
        "Mean VaR 2020": float(df_2020["VaR_pred"].mean()) if len(df_2020) else np.nan,
        "Max VaR 2020": float(df_2020["VaR_pred"].max()) if len(df_2020) else np.nan,
        "n_negative_var": int((df["VaR_pred"] < 0).sum()),
        "p_UC": kup["p_uc"],
        "UC Test": "PASS" if kup["p_uc"] > sig else "FAIL",
        "p_CC": cc["p_cc"],
        "CC Test": "PASS" if cc["p_cc"] > sig else "FAIL",
        "p_DQ": dq["p_dq"],
        "DQ Test": "PASS" if dq["p_dq"] > sig else "FAIL",
    }
    vals = [row["p_UC"], row["p_CC"], row["p_DQ"]]
    row["p_Mean"] = float(np.mean([v for v in vals if pd.notnull(v)]))
    row["Valid(CC&DQ)"] = "YES" if row["CC Test"] == "PASS" and row["DQ Test"] == "PASS" else "NO"
    return pd.DataFrame([row])


def evaluate_by_year(df, model_label, alpha=0.99):
    rows = []
    for year, g in df.groupby("year"):
        real_losses = g["loss_real"].values
        var_preds = g["VaR_pred"].values
        violations = int(g["viol"].sum())
        expected = (1 - alpha) * len(g)
        kup = kupiec_test(real_losses, var_preds, alpha=alpha)
        cc = christoffersen_cc_test(real_losses, var_preds, alpha=alpha)
        dq = dq_test(real_losses, var_preds, alpha=alpha, lags=4)
        rows.append({
            "Model": model_label,
            "year": int(year),
            "T": len(g),
            "Expected Viol.": expected,
            "Violations": violations,
            "Violation Rate": violations / len(g),
            "VR Gap": abs((violations / len(g)) - (1 - alpha)),
            "Tick Loss": tick_loss(real_losses, var_preds, alpha=alpha),
            "Winkler Score": winkler_score(real_losses, var_preds, alpha=alpha),
            "Mean VaR Level": float(np.nanmean(var_preds)),
            "Max VaR Level": float(np.nanmax(var_preds)),
            "Min VaR Level": float(np.nanmin(var_preds)),
            "Max Loss": float(np.nanmax(real_losses)),
            "Mean Loss": float(np.nanmean(real_losses)),
            "n_negative_var": int((g["VaR_pred"] < 0).sum()),
            "p_UC": kup["p_uc"],
            "UC Test": "PASS" if kup["p_uc"] > SIG else "FAIL",
            "p_CC": cc["p_cc"],
            "CC Test": "PASS" if cc["p_cc"] > SIG else "FAIL",
            "p_DQ": dq["p_dq"],
            "DQ Test": "PASS" if dq["p_dq"] > SIG else "FAIL",
        })
    return pd.DataFrame(rows)


def violation_table(df):
    cols = ["VaR_pred", "loss_real", "viol", "year"]
    out = df.loc[df["viol"] == 1, cols].copy()
    out["exceedance"] = out["loss_real"] - out["VaR_pred"]
    out["exceedance_ratio"] = out["loss_real"] / out["VaR_pred"]
    return out.sort_index()


def worst_days_table(df, n=25):
    out = df.copy()
    out["exceedance"] = out["loss_real"] - out["VaR_pred"]
    out["loss_minus_var"] = out["exceedance"]
    return out.sort_values("loss_real", ascending=False).head(n)


def violation_gap_summary(df):
    v = df.loc[df["viol"] == 1].copy()
    if len(v) < 2:
        return {"violations": len(v), "median_gap_days": np.nan, "gaps_le_5": 0, "gaps_le_20": 0}
    gaps = pd.Series(v.index).diff().dt.days.dropna()
    return {
        "violations": len(v),
        "median_gap_days": float(gaps.median()),
        "gaps_le_5": int((gaps <= 5).sum()),
        "gaps_le_20": int((gaps <= 20).sum()),
    }

## Modelos y entrenamiento

In [6]:
def inverse_softplus(y):
    return math.log(math.expm1(y))


# IMPORTANTE: no modificar. Misma perdida que los experimentos Softplus anteriores.
def var_loss(y_true, y_pred, q=0.99, w=1.0):
    e = y_true - y_pred
    weight = torch.where(e > 0, torch.as_tensor(w, dtype=y_pred.dtype, device=y_pred.device), torch.ones_like(e))
    pinball = torch.maximum(q * e, (q - 1) * e)
    return torch.mean(weight * pinball)


class MLPVaRSoftplusBase(nn.Module):
    def __init__(self, d_in):
        super().__init__()
        self.body = nn.Sequential(
            nn.Linear(d_in, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
        )
        self.softplus = nn.Softplus()
        nn.init.constant_(self.body[-1].bias, inverse_softplus(0.02))

    def forward(self, x):
        return self.softplus(self.body(x))


class MLPVaRSoftplusReLULayerNorm(nn.Module):
    def __init__(self, d_in):
        super().__init__()
        self.body = nn.Sequential(
            nn.Linear(d_in, 128),
            nn.LayerNorm(128),
            nn.ReLU(),
            nn.Dropout(0.10),
            nn.Linear(128, 64),
            nn.LayerNorm(64),
            nn.ReLU(),
            nn.Dropout(0.10),
            nn.Linear(64, 1),
        )
        self.softplus = nn.Softplus()
        nn.init.constant_(self.body[-1].bias, inverse_softplus(0.02))

    def forward(self, x):
        return self.softplus(self.body(x))


class MLPVaRSoftplusSiLULayerNorm(nn.Module):
    def __init__(self, d_in):
        super().__init__()
        self.body = nn.Sequential(
            nn.Linear(d_in, 128),
            nn.LayerNorm(128),
            nn.SiLU(),
            nn.Dropout(0.10),
            nn.Linear(128, 64),
            nn.LayerNorm(64),
            nn.SiLU(),
            nn.Dropout(0.10),
            nn.Linear(64, 1),
        )
        self.softplus = nn.Softplus()
        nn.init.constant_(self.body[-1].bias, inverse_softplus(0.02))

    def forward(self, x):
        return self.softplus(self.body(x))


def build_model(d_in, architecture):
    if architecture == "base_relu":
        return MLPVaRSoftplusBase(d_in=d_in)
    if architecture == "relu_layernorm":
        return MLPVaRSoftplusReLULayerNorm(d_in=d_in)
    if architecture == "silu_layernorm":
        return MLPVaRSoftplusSiLULayerNorm(d_in=d_in)
    raise ValueError(f"Arquitectura no soportada: {architecture}")


def train_one_model(X_train, y_train, d_in, seed, w_loss, architecture, alpha=0.99, max_epochs=200, lr=5e-5, batch_size=256, patience=15, val_split=0.10):
    torch.manual_seed(seed)
    np.random.seed(seed)
    n = len(X_train)
    split = int(n * (1 - val_split))
    X_tr, X_val = X_train[:split], X_train[split:]
    y_tr, y_val = y_train[:split], y_train[split:]
    model = build_model(d_in=d_in, architecture=architecture)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    train_loader = DataLoader(TensorDataset(torch.from_numpy(X_tr), torch.from_numpy(y_tr)), batch_size=batch_size, shuffle=False)
    X_val_t = torch.from_numpy(X_val)
    y_val_t = torch.from_numpy(y_val)
    best_loss = np.inf
    best_state = None
    patience_counter = 0
    for epoch in range(max_epochs):
        model.train()
        for xb, yb in train_loader:
            opt.zero_grad()
            pred = model(xb)
            loss_val = var_loss(yb, pred, q=alpha, w=w_loss)
            loss_val.backward()
            opt.step()
        model.eval()
        with torch.no_grad():
            val_pred = model(X_val_t)
            val_loss = var_loss(y_val_t, val_pred, q=alpha, w=w_loss).item()
        if val_loss < best_loss - 1e-4:
            best_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model

## Ejecucion de experimentos

In [7]:
def run_experiment(exp):
    eval_dates = dataset.loc[pd.Timestamp(EVAL_START):pd.Timestamp(EVAL_END)].index
    var_preds = []
    real_targets = []
    dates = []
    current_model = None
    muX, sdX = None, None

    desc = f"{exp['key']} w=1"
    for i, current_date in enumerate(tqdm(eval_dates, desc=desc, dynamic_ncols=True)):
        if i % RETRAIN_EVERY == 0:
            train_end = current_date - pd.Timedelta(days=1)
            train_df = dataset.loc[:train_end].tail(WINDOW)
            if len(train_df) < 250:
                if current_model is None:
                    continue
            else:
                X_train = train_df[feature_cols].values.astype(np.float32)
                y_train = train_df["target"].values.astype(np.float32).reshape(-1, 1)
                muX = X_train.mean(axis=0, keepdims=True)
                sdX = X_train.std(axis=0, keepdims=True) + 1e-8
                X_train = (X_train - muX) / sdX
                current_model = train_one_model(
                    X_train,
                    y_train,
                    d_in=X_train.shape[1],
                    seed=SEED,
                    w_loss=W_LOSS,
                    architecture=exp["architecture"],
                    alpha=ALPHA,
                )

        if current_model is None or muX is None or sdX is None:
            continue

        test_df = dataset.loc[[current_date]]
        X_test = (test_df[feature_cols].values.astype(np.float32) - muX) / sdX
        y_test = test_df["target"].values.astype(np.float32).reshape(-1, 1)

        current_model.eval()
        with torch.no_grad():
            pred = current_model(torch.from_numpy(X_test)).numpy().reshape(-1)[0]

        var_preds.append(float(pred))
        real_targets.append(float(y_test.reshape(-1)[0]))
        dates.append(current_date)

    pred_df = pd.DataFrame({"VaR_pred": var_preds, "loss_real": real_targets}, index=pd.DatetimeIndex(dates)).sort_index()
    pred_df = pred_df.loc[EVAL_START:EVAL_END].dropna().copy()
    pred_df["viol"] = (pred_df["loss_real"] > pred_df["VaR_pred"]).astype(int)
    pred_df["year"] = pred_df.index.year
    return pred_df


results = {}
summary_parts = []
yearly_parts = []
gap_parts = []

for exp in EXPERIMENTS:
    print(f"\n{'=' * 90}\nEjecutando: {exp['label']}\n{'=' * 90}")
    pred_df = run_experiment(exp)
    results[exp["key"]] = pred_df

    summary = evaluate_var_model(pred_df, model_label=exp["label"], alpha=ALPHA, sig=SIG)
    yearly = evaluate_by_year(pred_df, model_label=exp["label"], alpha=ALPHA)
    violations = violation_table(pred_df)
    gaps = violation_gap_summary(pred_df)
    gaps["Model"] = exp["label"]

    pred_path = DATA_DIR / exp["pred_file"]
    summary_path = DATA_DIR / f"{exp['prefix']}_summary.csv"
    yearly_path = DATA_DIR / f"{exp['prefix']}_yearly.csv"
    violations_path = DATA_DIR / f"{exp['prefix']}_violations.csv"

    pred_df.to_csv(pred_path)
    summary.to_csv(summary_path, index=False)
    yearly.to_csv(yearly_path, index=False)
    violations.to_csv(violations_path)

    print("Guardado:", pred_path)
    print("Guardado:", summary_path)
    print("Guardado:", yearly_path)
    print("Guardado:", violations_path)
    plausibility_report(pred_df)
    display(summary)
    display(yearly)
    display(violations)

    summary_parts.append(summary)
    yearly_parts.append(yearly)
    gap_parts.append(gaps)

new_summary = pd.concat(summary_parts, ignore_index=True)
new_yearly = pd.concat(yearly_parts, ignore_index=True)
gap_summary = pd.DataFrame(gap_parts)


Ejecutando: Softplus + dynamic features


dynamic_features w=1: 100%|██████████| 3762/3762 [16:36<00:00,  3.77it/s]

Guardado: ..\data\nn_softplus_dynamic_features_2010_2024_w1.csv
Guardado: ..\data\softplus_dynamic_features_2010_2024_w1_summary.csv
Guardado: ..\data\softplus_dynamic_features_2010_2024_w1_yearly.csv
Guardado: ..\data\softplus_dynamic_features_2010_2024_w1_violations.csv

loss_real
min: -0.07361457496881485
max: 0.07843015342950821
mean: -0.00016187175334081243
p95: 0.009858225146308528
p99: 0.017429363690316665
p99.9: 0.038501730956141275

VaR_pred
min: 0.013304420746862888
max: 0.07366997748613358
mean: 0.021152370334798463
p95: 0.025048439577221863
p99: 0.03249528355896473
p99.9: 0.053368298307062446
n_negative_var: 0
max VaR: 2020-03-30 00:00:00 0.07366997748613358
min VaR: 2020-03-24 00:00:00 0.013304420746862888


,Model,w,T,Expected Viol.,Violations,Violation Rate,VR Gap,Coverage Bias,Tick Loss,Winkler Score,Mean VaR Level,Max VaR Level,Min VaR Level,Mean VaR 2020,Max VaR 2020,n_negative_var,p_UC,UC Test,p_CC,CC Test,p_DQ,DQ Test,p_Mean,Valid(CC&DQ)
0,Softplus + dynamic features,1,3762,37.62,18,0.004785,0.005215,-0.005215,0.000269,0.021265,0.021152,0.07367,0.013304,0.024382,0.07367,0,0.000346,FAIL,0.00034,FAIL,1.059016e-07,FAIL,0.000228,NO


,Model,year,T,Expected Viol.,Violations,Violation Rate,VR Gap,Tick Loss,Winkler Score,Mean VaR Level,Max VaR Level,Min VaR Level,Max Loss,Mean Loss,n_negative_var,p_UC,UC Test,p_CC,CC Test,p_DQ,DQ Test
0,Softplus + dynamic features,2010,252,2.52,1,0.003968,0.006032,0.000236,0.020603,0.020550,0.025519,0.017133,0.026197,-0.000381,0,0.273177,PASS,0.546423,PASS,0.818540,PASS
1,Softplus + dynamic features,2011,252,2.52,3,0.011905,0.001905,0.000241,0.021039,0.020981,0.026943,0.015817,0.026702,-0.000277,0,0.767969,PASS,0.923289,PASS,0.998540,PASS
2,Softplus + dynamic features,2012,249,2.49,0,0.000000,0.010000,0.000206,0.020477,0.020477,0.028795,0.015293,0.019407,-0.000117,0,NaN,FAIL,NaN,FAIL,NaN,FAIL
3,Softplus + dynamic features,2013,251,2.51,2,0.007968,0.002032,0.000272,0.020722,0.020588,0.029680,0.015994,0.029816,0.000066,0,0.737311,PASS,0.930176,PASS,0.999324,PASS
4,Softplus + dynamic features,2014,252,2.52,1,0.003968,0.006032,0.000215,0.020231,0.020196,0.030683,0.015732,0.025232,0.000438,0,0.273177,PASS,0.546423,PASS,0.818540,PASS
5,Softplus + dynamic features,2015,252,2.52,1,0.003968,0.006032,0.000222,0.022538,0.022536,0.037241,0.016622,0.019679,0.000456,0,0.273177,PASS,0.546423,PASS,0.818540,PASS
6,Softplus + dynamic features,2016,250,2.50,0,0.000000,0.010000,0.000225,0.022061,0.022061,0.042512,0.017194,0.016855,-0.000410,0,NaN,FAIL,NaN,FAIL,NaN,FAIL
7,Softplus + dynamic features,2017,249,2.49,0,0.000000,0.010000,0.000207,0.020232,0.020232,0.023946,0.016210,0.014269,-0.000486,0,NaN,FAIL,NaN,FAIL,NaN,FAIL
8,Softplus + dynamic features,2018,250,2.50,0,0.000000,0.010000,0.000202,0.020521,0.020521,0.027598,0.015892,0.018189,0.000311,0,NaN,FAIL,NaN,FAIL,NaN,FAIL
9,Softplus + dynamic features,2019,251,2.51,0,0.000000,0.010000,0.000222,0.021613,0.021613,0.036381,0.014859,0.018302,-0.000611,0,NaN,FAIL,NaN,FAIL,NaN,FAIL


...

dynamic_silu_layernorm w=1: 100%|██████████| 3762/3762 [21:17<00:00,  2.95it/s]

Guardado: ..\data\nn_softplus_dynamic_silu_layernorm_2010_2024_w1.csv
Guardado: ..\data\softplus_dynamic_silu_layernorm_2010_2024_w1_summary.csv
Guardado: ..\data\softplus_dynamic_silu_layernorm_2010_2024_w1_yearly.csv
Guardado: ..\data\softplus_dynamic_silu_layernorm_2010_2024_w1_violations.csv

loss_real
min: -0.07361457496881485
max: 0.07843015342950821
mean: -0.00016187175334081243
p95: 0.009858225146308528
p99: 0.017429363690316665
p99.9: 0.038501730956141275

VaR_pred
min: 0.006577305495738983
max: 0.07084870338439941
mean: 0.02218865288980008
p95: 0.039264634065330026
p99: 0.05139642197638748
p99.9: 0.06457432471215754
n_negative_var: 0
max VaR: 2019-09-25 00:00:00 0.07084870338439941
min VaR: 2014-06-12 00:00:00 0.006577305495738983


,Model,w,T,Expected Viol.,Violations,Violation Rate,VR Gap,Coverage Bias,Tick Loss,Winkler Score,Mean VaR Level,Max VaR Level,Min VaR Level,Mean VaR 2020,Max VaR 2020,n_negative_var,p_UC,UC Test,p_CC,CC Test,p_DQ,DQ Test,p_Mean,Valid(CC&DQ)
0,Softplus + dynamic features + SiLU LayerNorm,1,3762,37.62,42,0.011164,0.001164,0.001164,0.000302,0.022348,0.022189,0.070849,0.006577,0.027367,0.06791,0,0.48109,PASS,0.618056,PASS,0.005804,FAIL,0.368316,NO


,Model,year,T,Expected Viol.,Violations,Violation Rate,VR Gap,Tick Loss,Winkler Score,Mean VaR Level,Max VaR Level,Min VaR Level,Max Loss,Mean Loss,n_negative_var,p_UC,UC Test,p_CC,CC Test,p_DQ,DQ Test
0,Softplus + dynamic features + SiLU LayerNorm,2010,252,2.52,3,0.011905,0.001905,0.000279,0.020005,0.019850,0.044727,0.008342,0.026197,-0.000381,0,0.767969,PASS,0.923289,PASS,0.998540,PASS
1,Softplus + dynamic features + SiLU LayerNorm,2011,252,2.52,3,0.011905,0.001905,0.000239,0.022405,0.022380,0.047493,0.008461,0.026702,-0.000277,0,0.767969,PASS,0.923289,PASS,0.998540,PASS
2,Softplus + dynamic features + SiLU LayerNorm,2012,249,2.49,4,0.016064,0.006064,0.000257,0.019604,0.019480,0.042337,0.008725,0.019407,-0.000117,0,0.376725,PASS,0.633651,PASS,0.008719,FAIL
3,Softplus + dynamic features + SiLU LayerNorm,2013,251,2.51,2,0.007968,0.002032,0.000301,0.020723,0.020527,0.050610,0.008492,0.029816,0.000066,0,0.737311,PASS,0.930176,PASS,0.999324,PASS
4,Softplus + dynamic features + SiLU LayerNorm,2014,252,2.52,1,0.003968,0.006032,0.000181,0.018105,0.018095,0.051497,0.006577,0.025232,0.000438,0,0.273177,PASS,0.546423,PASS,0.818540,PASS
5,Softplus + dynamic features + SiLU LayerNorm,2015,252,2.52,2,0.007937,0.002063,0.000299,0.027499,0.027440,0.059762,0.009787,0.019679,0.000456,0,0.732712,PASS,0.928317,PASS,0.999282,PASS
6,Softplus + dynamic features + SiLU LayerNorm,2016,250,2.50,4,0.016000,0.006000,0.000277,0.025970,0.025942,0.065038,0.009185,0.016855,-0.000410,0,0.380484,PASS,0.637706,PASS,0.971186,PASS
7,Softplus + dynamic features + SiLU LayerNorm,2017,249,2.49,1,0.004016,0.005984,0.000204,0.019064,0.019047,0.039483,0.008482,0.014269,-0.000486,0,0.280550,PASS,0.556404,PASS,0.831176,PASS
8,Softplus + dynamic features + SiLU LayerNorm,2018,250,2.50,4,0.016000,0.006000,0.000282,0.020746,0.020587,0.048990,0.008618,0.018189,0.000311,0,0.380484,PASS,0.637706,PASS,0.004472,FAIL
9,Softplus + dynamic features + SiLU LayerNorm,2019,251,2.51,2,0.007968,0.002032,0.000270,0.025439,0.025420,0.070849,0.008735,0.018302,-0.000611,0,0.737311,PASS,0.930176,PASS,0.999324,PASS


,VaR_pred,loss_real,viol,year,exceedance,exceedance_ratio
2010-02-03,0.014433,0.026197,1,2010,0.011764,1.815063
2010-05-03,0.014173,0.017654,1,2010,0.003481,1.245626
2010-10-18,0.013085,0.017095,1,2010,0.004010,1.306429
2011-05-10,0.013685,0.014313,1,2011,0.000628,1.045866
2011-08-05,0.012419,0.013372,1,2011,0.000952,1.076668
2011-09-27,0.013686,0.015219,1,2011,0.001534,1.112053
2012-06-20,0.012799,0.019407,1,2012,0.006607,1.516233
2012-07-05,0.009766,0.013480,1,2012,0.003714,1.380347
2012-11-01,0.010636,0.011887,1,2012,0.001251,1.117623
2012-11-06,0.009269,0.013013,1,2012,0.003744,1.403922


## Comparacion final

In [8]:
def load_prediction_csv(path, var_col="VaR_pred"):
    df = pd.read_csv(path, index_col=0, parse_dates=True).sort_index()
    df = df.loc[EVAL_START:EVAL_END].copy()
    if var_col != "VaR_pred":
        df = df.rename(columns={var_col: "VaR_pred"})
    if "viol" not in df.columns:
        df["viol"] = (df["loss_real"] > df["VaR_pred"]).astype(int)
    if "year" not in df.columns:
        df["year"] = df.index.year
    return df

comparison_rows = []
gap_rows = []

reference_files = [
    ("Softplus base w=1", DATA_DIR / "nn_softplus_validation_w1.csv", DATA_DIR / "nn_softplus_test_w1.csv"),
    ("Softplus + ratios + shocks", DATA_DIR / "nn_softplus_pruebas_shocks_2010_2024_w1.csv", None),
    ("Softplus + downside shocks + stress memory", DATA_DIR / "nn_softplus_pruebas_downside_stress_2010_2024_w1.csv", None),
]

for label, path_a, path_b in reference_files:
    if not path_a.exists():
        continue
    if path_b is not None and path_b.exists():
        ref_df = pd.concat([load_prediction_csv(path_a), load_prediction_csv(path_b)]).sort_index()
        ref_df = ref_df[~ref_df.index.duplicated(keep="last")].copy()
        ref_df = ref_df.loc[EVAL_START:EVAL_END]
    else:
        ref_df = load_prediction_csv(path_a)
    comparison_rows.append(evaluate_var_model(ref_df, model_label=label, alpha=ALPHA, sig=SIG))
    gaps = violation_gap_summary(ref_df)
    gaps["Model"] = label
    gap_rows.append(gaps)

comparison_rows.append(new_summary)
gap_rows.extend(gap_parts)

comparison = pd.concat(comparison_rows, ignore_index=True)
comparison_path = DATA_DIR / "softplus_dynamic_experiments_2010_2024_w1_comparison.csv"
comparison.to_csv(comparison_path, index=False)

gap_comparison = pd.DataFrame(gap_rows)
gap_path = DATA_DIR / "softplus_dynamic_experiments_2010_2024_w1_gap_comparison.csv"
gap_comparison.to_csv(gap_path, index=False)

print("Guardado:", comparison_path)
print("Guardado:", gap_path)

cols = [
    "Model", "w", "T", "Expected Viol.", "Violations", "Violation Rate", "VR Gap", "Coverage Bias",
    "Tick Loss", "Winkler Score", "Mean VaR Level", "Max VaR Level", "Min VaR Level",
    "Mean VaR 2020", "Max VaR 2020", "n_negative_var", "p_UC", "UC Test", "p_CC", "CC Test", "p_DQ", "DQ Test", "p_Mean", "Valid(CC&DQ)",
]

display(comparison[cols].sort_values(["DQ Test", "CC Test", "UC Test", "VR Gap", "Tick Loss"], ascending=[False, False, False, True, True]))

print("\nResumen de gaps entre violaciones")
display(gap_comparison[["Model", "violations", "median_gap_days", "gaps_le_5", "gaps_le_20"]])

print("\nComparacion anual de los nuevos experimentos")
display(new_yearly.sort_values(["Model", "year"]))

print("\nAnos problematicos de los nuevos experimentos")
display(new_yearly.sort_values(["Violation Rate", "Violations"], ascending=False).head(20))

Guardado: ..\data\softplus_dynamic_experiments_2010_2024_w1_comparison.csv
Guardado: ..\data\softplus_dynamic_experiments_2010_2024_w1_gap_comparison.csv


,Model,w,T,Expected Viol.,Violations,Violation Rate,VR Gap,Coverage Bias,Tick Loss,Winkler Score,Mean VaR Level,Max VaR Level,Min VaR Level,Mean VaR 2020,Max VaR 2020,n_negative_var,p_UC,UC Test,p_CC,CC Test,p_DQ,DQ Test,p_Mean,Valid(CC&DQ)
5,Softplus + dynamic features + SiLU LayerNorm,1,3762,37.62,42,0.011164,0.001164,0.001164,0.000302,0.022348,0.022189,0.070849,0.006577,0.027367,0.067910,0,0.481090,PASS,0.618056,PASS,5.803898e-03,FAIL,0.368316,NO
1,Softplus + ratios + shocks,1,3762,37.62,32,0.008506,0.001494,-0.001494,0.000274,0.018858,0.018684,0.028376,0.008985,0.019036,0.028376,0,0.344590,PASS,0.060605,PASS,0.000000e+00,FAIL,0.135065,NO
0,Softplus base w=1,1,3762,37.62,36,0.009569,0.000431,-0.000431,0.000283,0.018417,0.018217,0.030089,0.007403,0.017326,0.030089,0,0.789181,PASS,0.016972,FAIL,0.000000e+00,FAIL,0.268718,NO
2,Softplus + downside shocks + stress memory,1,3762,37.62,25,0.006645,0.003355,-0.003355,0.000283,0.020356,0.020195,0.030874,0.008058,0.020204,0.030874,0,0.027651,FAIL,0.032905,FAIL,0.000000e+00,FAIL,0.020185,NO
4,Softplus + dynamic features + ReLU LayerNorm,1,3762,37.62,21,0.005582,0.004418,-0.004418,0.000322,0.026124,0.026003,0.078225,0.007939,0.031421,0.078225,0,0.002967,FAIL,0.003305,FAIL,8.498885e-10,FAIL,0.002091,NO
3,Softplus + dynamic features,1,3762,37.62,18,0.004785,0.005215,-0.005215,0.000269,0.021265,0.021152,0.073670,0.013304,0.024382,0.073670,0,0.000346,FAIL,0.000340,FAIL,1.059016e-07,FAIL,0.000228,NO



Resumen de gaps entre violaciones


,Model,violations,median_gap_days,gaps_le_5,gaps_le_20
0,Softplus base w=1,36,35.0,8,16
1,Softplus + ratios + shocks,32,53.0,8,12
2,Softplus + downside shocks + stress memory,25,54.5,6,10
3,Softplus + dynamic features,18,104.0,2,3
4,Softplus + dynamic features + ReLU LayerNorm,21,140.5,3,6
5,Softplus + dynamic features + SiLU LayerNorm,42,68.0,5,11



Comparacion anual de los nuevos experimentos


,Model,year,T,Expected Viol.,Violations,Violation Rate,VR Gap,Tick Loss,Winkler Score,Mean VaR Level,Max VaR Level,Min VaR Level,Max Loss,Mean Loss,n_negative_var,p_UC,UC Test,p_CC,CC Test,p_DQ,DQ Test
0,Softplus + dynamic features,2010,252,2.52,1,0.003968,0.006032,0.000236,0.020603,0.020550,0.025519,0.017133,0.026197,-0.000381,0,0.273177,PASS,0.546423,PASS,0.818540,PASS
1,Softplus + dynamic features,2011,252,2.52,3,0.011905,0.001905,0.000241,0.021039,0.020981,0.026943,0.015817,0.026702,-0.000277,0,0.767969,PASS,0.923289,PASS,0.998540,PASS
2,Softplus + dynamic features,2012,249,2.49,0,0.000000,0.010000,0.000206,0.020477,0.020477,0.028795,0.015293,0.019407,-0.000117,0,NaN,FAIL,NaN,FAIL,NaN,FAIL
3,Softplus + dynamic features,2013,251,2.51,2,0.007968,0.002032,0.000272,0.020722,0.020588,0.029680,0.015994,0.029816,0.000066,0,0.737311,PASS,0.930176,PASS,0.999324,PASS
4,Softplus + dynamic features,2014,252,2.52,1,0.003968,0.006032,0.000215,0.020231,0.020196,0.030683,0.015732,0.025232,0.000438,0,0.273177,PASS,0.546423,PASS,0.818540,PASS
5,Softplus + dynamic features,2015,252,2.52,1,0.003968,0.006032,0.000222,0.022538,0.022536,0.037241,0.016622,0.019679,0.000456,0,0.273177,PASS,0.546423,PASS,0.818540,PASS
6,Softplus + dynamic features,2016,250,2.50,0,0.000000,0.010000,0.000225,0.022061,0.022061,0.042512,0.017194,0.016855,-0.000410,0,NaN,FAIL,NaN,FAIL,NaN,FAIL
7,Softplus + dynamic features,2017,249,2.49,0,0.000000,0.010000,0.000207,0.020232,0.020232,0.023946,0.016210,0.014269,-0.000486,0,NaN,FAIL,NaN,FAIL,NaN,FAIL
8,Softplus + dynamic features,2018,250,2.50,0,0.000000,0.010000,0.000202,0.020521,0.020521,0.027598,0.015892,0.018189,0.000311,0,NaN,FAIL,NaN,FAIL,NaN,FAIL
9,Softplus + dynamic features,2019,251,2.51,0,0.000000,0.010000,0.000222,0.021613,0.021613,0.036381,0.014859,0.018302,-0.000611,0,NaN,FAIL,NaN,FAIL,NaN,FAIL



Anos problematicos de los nuevos experimentos


,Model,year,T,Expected Viol.,Violations,Violation Rate,VR Gap,Tick Loss,Winkler Score,Mean VaR Level,Max VaR Level,Min VaR Level,Max Loss,Mean Loss,n_negative_var,p_UC,UC Test,p_CC,CC Test,p_DQ,DQ Test
40,Softplus + dynamic features + SiLU LayerNorm,2020,251,2.51,8,0.031873,0.021873,0.000955,0.028729,0.027367,0.067910,0.008527,0.078430,-0.000757,0,0.005557,FAIL,0.010695,FAIL,0.099240,PASS
10,Softplus + dynamic features,2020,251,2.51,6,0.023904,0.013904,0.000882,0.025656,0.024382,0.073670,0.013304,0.078430,-0.000757,0,0.060378,PASS,0.050860,PASS,0.022358,FAIL
25,Softplus + dynamic features + ReLU LayerNorm,2020,251,2.51,6,0.023904,0.013904,0.000944,0.032678,0.031421,0.078225,0.009741,0.078430,-0.000757,0,0.060378,PASS,0.050860,PASS,0.022358,FAIL
32,Softplus + dynamic features + SiLU LayerNorm,2012,249,2.49,4,0.016064,0.006064,0.000257,0.019604,0.019480,0.042337,0.008725,0.019407,-0.000117,0,0.376725,PASS,0.633651,PASS,0.008719,FAIL
36,Softplus + dynamic features + SiLU LayerNorm,2016,250,2.50,4,0.016000,0.006000,0.000277,0.025970,0.025942,0.065038,0.009185,0.016855,-0.000410,0,0.380484,PASS,0.637706,PASS,0.971186,PASS
38,Softplus + dynamic features + SiLU LayerNorm,2018,250,2.50,4,0.016000,0.006000,0.000282,0.020746,0.020587,0.048990,0.008618,0.018189,0.000311,0,0.380484,PASS,0.637706,PASS,0.004472,FAIL
42,Softplus + dynamic features + SiLU LayerNorm,2022,251,2.51,4,0.015936,0.005936,0.000304,0.024671,0.024545,0.051332,0.009210,0.026797,0.000322,0,0.384255,PASS,0.641744,PASS,0.971831,PASS
17,Softplus + dynamic features + ReLU LayerNorm,2012,249,2.49,3,0.012048,0.002048,0.000266,0.023692,0.023634,0.054914,0.009578,0.019407,-0.000117,0,0.752993,PASS,0.917363,PASS,0.998349,PASS
23,Softplus + dynamic features + ReLU LayerNorm,2018,250,2.50,3,0.012000,0.002000,0.000296,0.023578,0.023447,0.061119,0.009054,0.018189,0.000311,0,0.757988,PASS,0.919379,PASS,0.000002,FAIL
12,Softplus + dynamic features,2022,251,2.51,3,0.011952,0.001952,0.000238,0.021565,0.021513,0.032491,0.016485,0.026797,0.000322,0,0.762980,PASS,0.921355,PASS,0.998479,PASS


## Lectura esperada

Un experimento solo es candidato si mejora el fallo del modelo 2 sin destruir la cobertura global:

- objetivo principal: `UC`, `CC` y `DQ` globales en `PASS`;
- si `DQ` no pasa, mirar si sube mucho `p_DQ` frente al modelo 2;
- la tasa de violacion deberia quedar cerca del 1%;
- el VaR medio no debe subir de forma generalizada;
- marzo de 2020 y 2022 son los anos clave para diagnosticar clustering.